# 04 — Feature Engineering

## Goal
Derive behavioural risk features from raw history (utilization, delinquency, payment ratios, trends).

In [ ]:
from pathlib import Path
import sys

# Locate the project root (works whether the notebook is run from notebooks/ or root)
root = Path.cwd()
while not (root / "src" / "data_loader.py").exists() and root != root.parent:
    root = root.parent
sys.path.insert(0, str(root))
print("project root:", root)


In [ ]:
from src.data_loader import load_clean, TARGET
from src.feature_engineering import add_features, feature_columns, get_feature_groups

df = add_features(load_clean())
new_cols = [c for c in df.columns if c not in feature_columns(df) and c != TARGET]
print(f"{len(feature_columns(df))} model features, e.g.:")
print(feature_columns(df)[-12:])


In [ ]:
df[["UTILIZATION_1", "UTILIZATION_MAX", "LATE_PAYMENT_COUNT_6M",
    "MAX_DELINQUENCY_6M", "WEIGHTED_DELINQUENCY", "PAY_TO_BILL_RATIO",
    "BILL_TREND"]].describe().T.round(3)


In [ ]:
feats = [c for c in feature_columns(df) if not c.startswith(("PAY_", "BILL_AMT", "SEX", "EDUCATION", "MARRIAGE", "LIMIT", "AGE"))]
corr = df[feats + [TARGET]].corr()[TARGET].drop(TARGET).sort_values(key=abs, ascending=False)
corr.round(3).head(12)


Engineered behavioural features (utilization, delinquency counts, recency-weighted delay) rank among the strongest linear correlates of default — and later dominate the SHAP importance ranking.